In [30]:
import torch
import torch.nn as nn
from torch import tensor
import torch.nn.functional as F

### Paremeters

In [10]:
attention = nn.MultiheadAttention(
    embed_dim=10,
    num_heads=1,
    batch_first=True,
    bias=False
)

In [11]:
for name, p in attention.named_parameters():
    print(name, ":", p.shape)

in_proj_weight : torch.Size([30, 10])
out_proj.weight : torch.Size([10, 10])


### Simple Call

In [40]:
attention = nn.MultiheadAttention(
    embed_dim=1,
    num_heads=1,
    batch_first=True,
    bias=False
)
attention.load_state_dict({
    "in_proj_weight": torch.tensor([
        [1.0],
        [2.0],
        [3.0],
    ]),
    "out_proj.weight": torch.tensor([
        [4.0],
    ])
})

<All keys matched successfully>

In [41]:
for name, p in attention.named_parameters():
    print(name, ":", p)

in_proj_weight : Parameter containing:
tensor([[1.],
        [2.],
        [3.]], requires_grad=True)
out_proj.weight : Parameter containing:
tensor([[4.]], requires_grad=True)


In [42]:
x = torch.tensor([
    [
        [0.2],
    ]
])

In [43]:
x.shape

torch.Size([1, 1, 1])

In [44]:
output, weights = attention(
    query=x,
    key=x,
    value=x
)

In [45]:
output.shape, weights.shape

(torch.Size([1, 1, 1]), torch.Size([1, 1, 1]))

In [46]:
output, weights

(tensor([[[2.4000]]], grad_fn=<TransposeBackward0>),
 tensor([[[1.]]], grad_fn=<MeanBackward1>))

In [47]:
x = torch.tensor(0.2)

W_Q = torch.tensor(1.0)
W_K = torch.tensor(2.0)
W_V = torch.tensor(3.0)

W_O = torch.tensor(4.0)

Q = x * W_Q
K = x * W_K
V = x * W_V

score = Q * K
d_k = 1
score_scaled = score / torch.sqrt(torch.tensor(d_k))

attention_weight = F.softmax(
    score_scaled.unsqueeze(0),
    dim=0
)

attention_output = attention_weight * V
output = attention_output * W_O

In [48]:
output

tensor([2.4000])

In [49]:
attention_weight

tensor([1.])

In [50]:
score

tensor(0.0800)

In [51]:
score_scaled

tensor(0.0800)

## Seq > 1

In [52]:
attention = nn.MultiheadAttention(
    embed_dim=1,
    num_heads=1,
    batch_first=True,
    bias=False
)
attention.load_state_dict({
    "in_proj_weight": torch.tensor([
        [1.0],
        [2.0],
        [3.0],
    ]),
    "out_proj.weight": torch.tensor([
        [4.0],
    ])
})

x = torch.tensor([
    [
        [0.2],
        [0.3],
    ]
])
x.shape

torch.Size([1, 2, 1])

In [53]:
output, weights = attention(
    query=x,
    key=x,
    value=x
)
output, weights

(tensor([[[3.0120],
          [3.0180]]], grad_fn=<TransposeBackward0>),
 tensor([[[0.4900, 0.5100],
          [0.4850, 0.5150]]], grad_fn=<MeanBackward1>))

In [54]:
output.shape, weights.shape

(torch.Size([1, 2, 1]), torch.Size([1, 2, 2]))

In [59]:
x = torch.tensor([
    [
        [0.2],
        [0.3],
    ]
])

W_Q = torch.tensor([[1.0]])
W_K = torch.tensor([[2.0]])
W_V = torch.tensor([[3.0]])

W_O = torch.tensor([[4.0]])

# Q, K, V
Q = x @ W_Q
K = x @ W_K
V = x @ W_V

score = Q @ K.transpose(-2, -1)
score_scaled = score / torch.sqrt(torch.tensor(1.0))

weights = F.softmax(score_scaled, dim=-1)
attn = weights @ V
output = attn @ W_O

In [60]:
output.shape, weights.shape

(torch.Size([1, 2, 1]), torch.Size([1, 2, 2]))

In [61]:
output, weights

(tensor([[[3.0120],
          [3.0180]]]),
 tensor([[[0.4900, 0.5100],
          [0.4850, 0.5150]]]))

In [62]:
score

tensor([[[0.0800, 0.1200],
         [0.1200, 0.1800]]])

In [64]:
weights

tensor([[[0.4900, 0.5100],
         [0.4850, 0.5150]]])

### Embedding Size > 1

In [66]:
attention = nn.MultiheadAttention(
    embed_dim=2,
    num_heads=1,
    batch_first=True,
    bias=False
)

attention.load_state_dict({
    "in_proj_weight": torch.tensor([
        # W_Q
        [0.1, 0.2],
        [0.3, 0.4],

        # W_K
        [0.2, 0.3],
        [0.4, 0.5],

        # W_V
        [0.3, 0.4],
        [0.5, 0.6],
    ]),
    
    "out_proj.weight": torch.tensor([
        [1.0, 1.1],
        [1.2, 1.3],
    ])
})

x = torch.tensor([
    [
        [0.2, 0.3],
        [0.4, 0.5],
    ]
])
x.shape

torch.Size([1, 2, 2])

In [67]:
output, weights = attention(
    query=x,
    key=x,
    value=x
)
output, weights

(tensor([[[0.6817, 0.8102],
          [0.6838, 0.8127]]], grad_fn=<TransposeBackward0>),
 tensor([[[0.4929, 0.5071],
          [0.4873, 0.5127]]], grad_fn=<MeanBackward1>))

In [72]:
x = torch.tensor([
    [
        [0.2, 0.3],
        [0.4, 0.5],
    ]
])

W_Q = torch.tensor([
    [0.1, 0.2],
    [0.3, 0.4],
])

W_K = torch.tensor([
    [0.2, 0.3],
    [0.4, 0.5],
])

W_V = torch.tensor([
    [0.3, 0.4],
    [0.5, 0.6],
])

W_O = torch.tensor([
    [1.0, 1.1],
    [1.2, 1.3],
])

Q = x @ W_Q.T
K = x @ W_K.T
V = x @ W_V.T

score = Q @ K.transpose(-2, -1)

score_scaled = score / torch.sqrt(torch.tensor(2.0))

weights = F.softmax(score_scaled, dim=-1)

attn = weights @ V

output = attn @ W_O.T

In [73]:
output, weights

(tensor([[[0.6817, 0.8102],
          [0.6838, 0.8127]]]),
 tensor([[[0.4929, 0.5071],
          [0.4873, 0.5127]]]))